# 06. Leaf check: cut-off for the input check, measured on validation leaves only
The API refuses a photo before the model when too little of it has plant colour (`api/leaf_check.py`). This notebook sets that cut-off.
1. Plant fraction for every validation image, with the same function the API uses (downloaded from the repository, not copied).
2. Cut-off = 1st percentile of the plant fraction on the validation fit half (the same stratified half as notebook 04b), so by construction about 1 in 100 real leaves would be refused there.
3. Measured on the other half: how many real leaves are refused, overall and per class (diseased leaves are yellower, so a class can be refused more often).
4. The test split and CCMT are not read. Runs on CPU; no GPU needed.

In [2]:
# Cell 1: environment, Drive, Kaggle
from google.colab import drive
drive.mount("/content/drive")
!pip install --quiet --upgrade kaggle scikit-learn
import os, json, numpy as np, pandas as pd
from PIL import Image
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")  # Colab Secrets, never in the notebook
!mkdir --parents ~/.kaggle && echo "$KAGGLE_API_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
RESULTS_DIRECTORY = "/content/drive/MyDrive/cassacheck_results"
os.makedirs(RESULTS_DIRECTORY, exist_ok=True)
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "healthy"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Cell 2: validation images only, and the API's own plant_fraction function
!kaggle competitions download --competition cassava-leaf-disease-classification --path /content/data
!unzip -q -o /content/data/cassava-leaf-disease-classification.zip "train_images/*" -d /content/data
!wget --quiet --output-document /content/split_v1_seed42.csv https://raw.githubusercontent.com/Justine-abc/cassavacheck/main/data/split_v1_seed42.csv
!wget --quiet --output-document /content/leaf_check.py https://raw.githubusercontent.com/Justine-abc/cassavacheck/main/api/leaf_check.py
import sys; sys.path.insert(0, "/content")
from leaf_check import plant_fraction, HUE_RANGE_DEGREES, MIN_SATURATION, MIN_VALUE
labels = pd.read_csv("/content/split_v1_seed42.csv")
validation_frame = labels[labels.split == "validation"].reset_index(drop=True)
print("validation", len(validation_frame), "| hue", HUE_RANGE_DEGREES, "| saturation >=", MIN_SATURATION, "| value >=", MIN_VALUE)

100% 5.76G/5.76G [01:10<00:00, 87.6MB/s]

validation 3210 | hue (35, 165) | saturation >= 0.15 | value >= 0.12


In [4]:
# Cell 3: plant fraction for every validation image
validation_frame["plant_fraction"] = [plant_fraction(Image.open(f"/content/data/train_images/{name}")) for name in validation_frame.image_id]
validation_frame[["image_id", "class", "plant_fraction"]].to_csv(f"{RESULTS_DIRECTORY}/leaf_check_validation_fractions.csv", index=False)
print(validation_frame.groupby("class").plant_fraction.describe(percentiles=[0.01, 0.05, 0.5]).round(3))

          count   mean    std    min     1%     5%    50%    max
class                                                           
CBB       164.0  0.727  0.177  0.194  0.296  0.414  0.749  0.998
CBSD      330.0  0.711  0.205  0.076  0.097  0.308  0.759  0.999
CGM       357.0  0.706  0.187  0.061  0.272  0.359  0.748  0.997
CMD      1972.0  0.730  0.182  0.081  0.226  0.380  0.765  1.000
healthy   387.0  0.706  0.195  0.102  0.182  0.364  0.734  0.999


In [5]:
# Cell 4: cut-off on the fit half (same stratified half as notebook 04b), refusals counted on the measure half
from sklearn.model_selection import train_test_split
fit_index, measure_index = train_test_split(np.arange(len(validation_frame)), test_size=0.5, stratify=validation_frame["label"], random_state=42)
fit, measure = validation_frame.iloc[fit_index], validation_frame.iloc[measure_index]
cut_off = round(float(np.percentile(fit.plant_fraction, 1)), 4)
refused = measure[measure.plant_fraction < cut_off]
print(f"cut-off (1st percentile, fit half, {len(fit)} images): {cut_off}")
print(f"measure half: {len(refused)} of {len(measure)} real leaves refused")
print("per class, refused / images:")
print(pd.DataFrame({"refused": refused["class"].value_counts(), "images": measure["class"].value_counts()}).reindex(CLASS_NAMES).fillna(0).astype(int))

cut-off (1st percentile, fit half, 1605 images): 0.1764
measure half: 8 of 1605 real leaves refused
per class, refused / images:
         refused  images
class                   
CBB            0      82
CBSD           3     165
CGM            0     179
CMD            4     986
healthy        1     193


In [6]:
# Cell 5: the frozen value for the repository; download /content/leaf_check.json into models/
json.dump({"min_plant_fraction": cut_off,
           "rule": "1st percentile of plant fraction on the validation fit half",
           "fitted_on": "validation split, stratified fit half (1,605 images), seed 42",
           "measured_on": f"validation split, other half: {len(refused)} of {len(measure)} real leaves refused"},
          open("/content/leaf_check.json", "w"), indent=2)
print(open("/content/leaf_check.json").read())

{
  "min_plant_fraction": 0.1764,
  "rule": "1st percentile of plant fraction on the validation fit half",
  "fitted_on": "validation split, stratified fit half (1,605 images), seed 42",
  "measured_on": "validation split, other half: 8 of 1605 real leaves refused"
}
